In [29]:

import argparse
import hashlib
import json
from itertools import product
from pathlib import Path
from itertools import product
import numpy as np

### Setup

In [30]:
SPECTRAL_INDICES = [ 2.0, 2.5, 2.7]

FIXED = {
    "signal_injection_gamma": 2.5,
    "dpsi_nbins": 101,
    "weight_field": "oneweight",
    "true_ra_name": "true_ra",
    "true_dec_name": "true_dec",
    "true_energy_name": "true_energy",
    "angular_cutoff_deg": 15.0,
}


ENERGY_EDGE_CONFIGS = [
    [2.0, 2.5, 3., 3.5, 4, 4.5, 5., 6., 10.],
    [2.0, 2.5, 3, 3.25, 3.5, 4.25, 5., 6.0, 10.],
    [2, 3, 4, 5, 6, 10.],
    [2, 2.5, 3.25, 4.25, 6.0, 10.]
]


SINDEC_EDGE_CONFIGS = [
    np.linspace(-1, 1, 5),
    np.linspace(-1, 1, 10),
    np.linspace(-1, 1, 15),
    np.linspace(-1, 1, 25)
]


SIGMAS = [10, 12, 15, 20]

MINIMUM_COUNTS = [300]



### Defintitions

In [31]:
def canonicalize_edges(edges, precision=8):
    return [round(float(x), precision) for x in edges]


def short_hash(values, digits=6):
    values = canonicalize_edges(values)
    payload = json.dumps(values, separators=(",", ":"), sort_keys=True)
    return hashlib.sha1(payload.encode("utf-8")).hexdigest()[:digits]


def make_candidate_id(log10energy, dec_sindec_edges, sigma, minimum_counts):
    n_energy_bins = len(log10energy) - 1
    n_sindec_bins = len(dec_sindec_edges) - 1

    energy_hash = short_hash(log10energy)
    sindec_hash = short_hash(dec_sindec_edges)

    return (
        f"E{n_energy_bins}_{energy_hash}_"
        f"D{n_sindec_bins}_{sindec_hash}_"
        f"S{int(sigma):02d}_MC{int(minimum_counts)}"
    )


def make_candidate(log10energy, dec_sindec_edges, sigma, minimum_counts):
    return {
        "id": make_candidate_id(
            log10energy=log10energy,
            dec_sindec_edges=dec_sindec_edges,
            sigma=sigma,
            minimum_counts=minimum_counts,
        ),
        "minimum_counts": int(minimum_counts),
        "parametrization_bins": {
            "log10energy": canonicalize_edges(log10energy),
            "dec_sindec_edges": canonicalize_edges(dec_sindec_edges),
            "sigma": int(sigma),
        },
    }


import random
from itertools import product


def _check_unique_ids(candidates):
    ids = [candidate["id"] for candidate in candidates]
    if len(ids) != len(set(ids)):
        raise ValueError("Duplicate candidate IDs detected.")


def _homogeneous_downsample(combinations, target_size, seed=0):
    """
    Deterministically samples the full parameter-space list as evenly as possible.

    This assumes `combinations` is ordered Cartesian-product output.
    It picks evenly spaced indices across the full list and optionally jitters
    within each interval using `seed`.
    """
    n_total = len(combinations)

    if target_size is None or target_size >= n_total:
        return combinations

    if target_size <= 0:
        raise ValueError("target_size must be positive.")

    rng = random.Random(seed)

    sampled = []
    used_indices = set()

    for i in range(target_size):
        start = i * n_total / target_size
        end = (i + 1) * n_total / target_size

        idx_min = int(start)
        idx_max = max(idx_min, int(end) - 1)

        idx = rng.randint(idx_min, idx_max)

        while idx in used_indices:
            idx = (idx + 1) % n_total

        used_indices.add(idx)
        sampled.append(combinations[idx])

    return sampled


def build_all_combinations(target_size=None, seed=0):
    combinations = list(
        product(
            ENERGY_EDGE_CONFIGS,
            SINDEC_EDGE_CONFIGS,
            SIGMAS,
            MINIMUM_COUNTS,
        )
    )

    combinations = _homogeneous_downsample(
        combinations=combinations,
        target_size=target_size,
        seed=seed,
    )

    candidates = [
        make_candidate(
            log10energy=log10energy,
            dec_sindec_edges=dec_sindec_edges,
            sigma=sigma,
            minimum_counts=minimum_counts,
        )
        for log10energy, dec_sindec_edges, sigma, minimum_counts in combinations
    ]

    _check_unique_ids(candidates)

    return candidates

### Execution

for `build_all_combinations` one can choose a target size of configurations if desired, which then applies homogenous downsampling to the dataset

In [32]:
OUTPUT_FILE = "candidates.json"
INDENT = 2
candidates = build_all_combinations( seed = 0)
print(f'Number of candidates: {len(candidates)}')
print(f'Full scan would be: {len(build_all_combinations(seed = 0))}')
config = {
    "spectral_indices": SPECTRAL_INDICES,
    "fixed": FIXED,
    "candidates": candidates,
}


Number of candidates: 64
Full scan would be: 64


In [33]:

with open(OUTPUT_FILE, "w") as f:
    json.dump(config, f, indent=INDENT)

print(f"Wrote {len(config['candidates'])} candidates to {OUTPUT_FILE}")
config["candidates"][:3]

Wrote 64 candidates to candidates.json


[{'id': 'E8_b26d8e_D4_fc5cd0_S10_MC300',
  'minimum_counts': 300,
  'parametrization_bins': {'log10energy': [2.0,
    2.5,
    3.0,
    3.5,
    4.0,
    4.5,
    5.0,
    6.0,
    10.0],
   'dec_sindec_edges': [-1.0, -0.5, 0.0, 0.5, 1.0],
   'sigma': 10}},
 {'id': 'E8_b26d8e_D4_fc5cd0_S12_MC300',
  'minimum_counts': 300,
  'parametrization_bins': {'log10energy': [2.0,
    2.5,
    3.0,
    3.5,
    4.0,
    4.5,
    5.0,
    6.0,
    10.0],
   'dec_sindec_edges': [-1.0, -0.5, 0.0, 0.5, 1.0],
   'sigma': 12}},
 {'id': 'E8_b26d8e_D4_fc5cd0_S15_MC300',
  'minimum_counts': 300,
  'parametrization_bins': {'log10energy': [2.0,
    2.5,
    3.0,
    3.5,
    4.0,
    4.5,
    5.0,
    6.0,
    10.0],
   'dec_sindec_edges': [-1.0, -0.5, 0.0, 0.5, 1.0],
   'sigma': 15}}]

### Create corresponding list for bkg trials submission file:

In [34]:

SINDECS = [-0.5, -0.3, -0.1, 0.0, 0.2, 0.4]
NTRIALS = [1000]
CANDIDATE_IDS = [c["id"] for c in config["candidates"]]

print("queue SINDEC, NTRIALS, CANDIDATE_ID from (")

for sindec, ntrials, candidate_id in product(
    SINDECS,
    NTRIALS,
    CANDIDATE_IDS,
):
    print(f"{sindec} {ntrials} {candidate_id}")

print(")")

queue SINDEC, NTRIALS, CANDIDATE_ID from (
-0.5 1000 E8_b26d8e_D4_fc5cd0_S10_MC300
-0.5 1000 E8_b26d8e_D4_fc5cd0_S12_MC300
-0.5 1000 E8_b26d8e_D4_fc5cd0_S15_MC300
-0.5 1000 E8_b26d8e_D4_fc5cd0_S20_MC300
-0.5 1000 E8_b26d8e_D9_83b58e_S10_MC300
-0.5 1000 E8_b26d8e_D9_83b58e_S12_MC300
-0.5 1000 E8_b26d8e_D9_83b58e_S15_MC300
-0.5 1000 E8_b26d8e_D9_83b58e_S20_MC300
-0.5 1000 E8_b26d8e_D14_f505b9_S10_MC300
-0.5 1000 E8_b26d8e_D14_f505b9_S12_MC300
-0.5 1000 E8_b26d8e_D14_f505b9_S15_MC300
-0.5 1000 E8_b26d8e_D14_f505b9_S20_MC300
-0.5 1000 E8_b26d8e_D24_f8ce7a_S10_MC300
-0.5 1000 E8_b26d8e_D24_f8ce7a_S12_MC300
-0.5 1000 E8_b26d8e_D24_f8ce7a_S15_MC300
-0.5 1000 E8_b26d8e_D24_f8ce7a_S20_MC300
-0.5 1000 E8_a408dd_D4_fc5cd0_S10_MC300
-0.5 1000 E8_a408dd_D4_fc5cd0_S12_MC300
-0.5 1000 E8_a408dd_D4_fc5cd0_S15_MC300
-0.5 1000 E8_a408dd_D4_fc5cd0_S20_MC300
-0.5 1000 E8_a408dd_D9_83b58e_S10_MC300
-0.5 1000 E8_a408dd_D9_83b58e_S12_MC300
-0.5 1000 E8_a408dd_D9_83b58e_S15_MC300
-0.5 1000 E8_a408dd_D9_83b58e

In [35]:
num_candidates_tot = len(SINDECS)* len(NTRIALS)* len(CANDIDATE_IDS)
print(f'#Candidates (total): {num_candidates_tot}')
time_per_candidate_hours = 0.3 
print(f'Est. runtime @32cpu (h): {num_candidates_tot* time_per_candidate_hours}') 
print(f'Est. runtime @32cpu (d): {num_candidates_tot* time_per_candidate_hours /24}') 

#Candidates (total): 384
Est. runtime @32cpu (h): 115.19999999999999
Est. runtime @32cpu (d): 4.8


In [36]:
import os 
for c in CANDIDATE_IDS: 
    os.makedirs(f'/data/user/fkrafft/king_llh_binning_run/{c}/log', exist_ok= True)

In [37]:
len(CANDIDATE_IDS)

64